# XLM-ROBERTA Finnish Sentiment Notebook

## Imports

In [1]:
from datetime import datetime

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
import datetime
import numpy as np
import torch

## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## GPU/RAM Check

In [3]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Mar 19 11:41:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


## Load Model

In [5]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"Current timestamp: {timestamp}")

Current timestamp: 2026-03-19_11-41-38


In [6]:
BASE_MODEL = 'FacebookAI/xlm-roberta-large'
FINNSENTIMENT_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_{timestamp}/'
NUM_LABELS  = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

print(f"Base model loaded: {BASE_MODEL}")
print(f"Model device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model loaded: FacebookAI/xlm-roberta-large
Model device: cuda:0


## Preparing Training Data

In [7]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [8]:
ds.sample(5)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
291,Inderes.2327,Inderes.67,Mitä tästä pitäisi sanoa…? Tällä kertaa myyjän...,2018-09-20 14:59:06.764,3,Inderes,https://forum.inderes.com/t/admicom-rakennusal...,Admicom,ADMCM,71,2018-09,2018,0,2026-03-16T19:15:05.303899
123,Kauppalehti.post-6067248,Kauppalehti.41081,> Ketju on selvästikin jo häipynyt kesälaitumi...,2019-06-03 16:28:11.000,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/reve...,Revenio,REG1V,462,2019-06,2019,1,2026-03-16T17:22:23.344833
151,Inderes.16914,Inderes.406,Onkohan jakelulisenssiliikevaihdon osuutta ja ...,2019-07-30 06:48:03.458,4,Inderes,https://forum.inderes.com/t/qt-group-kasvurake...,Qt Group,QTCOM,245,2019-07,2019,1,2026-03-16T17:32:50.509459
115,Sijoitustieto.comment-8010,Sijoitustieto.Unknown,Ostaisin pää märkänä jos ei olisi jo isolla pa...,2016-04-05 14:37:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Nordea,NDA-FI,275,2016-04,2016,2,2026-03-16T17:20:24.331651
311,Kauppalehti.post-3754481,Kauppalehti.51897,"Kurssireaktio on varsin maltillinen, verraten ...",2019-08-08 08:27:42.000,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/capm...,Capman,CAPMAN,482,2019-08,2019,2,2026-03-16T19:23:39.512639


In [9]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## Fine-tune on Financial Domain Data

In [10]:
# 80/20 stratified split — holdout never seen during fine-tuning
ft_train_df, ft_holdout_df = train_test_split(
    ds[["message", "sentiment", "company_name"]],
    test_size=0.2,
    random_state=42,
    stratify=ds["sentiment"],
)

print(f"Fine-tune train: {len(ft_train_df)}  |  Holdout: {len(ft_holdout_df)}")
print("Train distribution:\n", ft_train_df["sentiment"].value_counts().sort_index())
print("Holdout distribution:\n", ft_holdout_df["sentiment"].value_counts().sort_index())

Fine-tune train: 486  |  Holdout: 122
Train distribution:
 sentiment
0    120
1    202
2    164
Name: count, dtype: int64
Holdout distribution:
 sentiment
0    30
1    51
2    41
Name: count, dtype: int64


In [11]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

ft_train_ds   = make_hf_dataset(ft_train_df)
ft_holdout_ds = make_hf_dataset(ft_holdout_df)

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

In [12]:
from torch import nn
from sklearn.utils.class_weight import compute_class_weight

FINETUNED_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_finetuned_{timestamp}/'

# Compute class weights from the training split to counter neutral-class dominance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=ft_train_df['sentiment'].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(model.device)
print(f"Class weights: neg={class_weights[0]:.3f}  neu={class_weights[1]:.3f}  pos={class_weights[2]:.3f}")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

ft_training_args = TrainingArguments(
    output_dir='/tmp/ft_checkpoints/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,           # 10% of total steps — more robust than a fixed count
    logging_steps=5,            # small dataset: log every 5 steps to see training loss
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

ft_trainer = WeightedTrainer(
    model=model,
    args=ft_training_args,
    train_dataset=ft_train_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

ft_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Class weights: neg=1.350  neu=0.802  pos=0.988


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.115562,1.100262,0.303279,0.159416,0.108125,0.303279
2,1.116032,1.100236,0.303279,0.208718,0.168730,0.303279
3,1.096299,1.098841,0.336066,0.169064,0.112940,0.336066
4,1.104650,1.099347,0.327869,0.207103,0.182279,0.327869
5,1.106138,1.097908,0.327869,0.166989,0.112022,0.327869
6,1.113898,1.097728,0.311475,0.204214,0.172382,0.311475
7,1.101425,1.097850,0.336066,0.292513,0.450066,0.336066
8,1.091143,1.097909,0.336066,0.218136,0.195036,0.336066
9,1.086127,1.097867,0.344262,0.236151,0.406206,0.344262
10,1.089375,1.097977,0.344262,0.236151,0.406206,0.344262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=310, training_loss=1.1023621143833284, metrics={'train_runtime': 118.2473, 'train_samples_per_second': 41.1, 'train_steps_per_second': 2.622, 'total_flos': 3495289554750636.0, 'train_loss': 1.1023621143833284, 'epoch': 10.0})

In [13]:
# Save fine-tuned model
ft_trainer.save_model(FINETUNED_MODEL_PATH)
tokenizer.save_pretrained(FINETUNED_MODEL_PATH)
print(f"Fine-tuned model saved to: {FINETUNED_MODEL_PATH}")

# Evaluate on holdout
holdout_results = ft_trainer.predict(ft_holdout_ds)
ft_preds = np.argmax(holdout_results.predictions, axis=1)
ft_true = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS AFTER FINE-TUNING")
print("=" * 50)
print(classification_report(ft_true, ft_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, ft_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_finetuned_2026-03-19_11-41-38/



HOLDOUT RESULTS AFTER FINE-TUNING
              precision    recall  f1-score   support

    negative       0.21      0.23      0.22        30
     neutral       0.80      0.08      0.14        51
    positive       0.35      0.71      0.46        41

    accuracy                           0.33       122
   macro avg       0.45      0.34      0.28       122
weighted avg       0.50      0.33      0.27       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative              7             1             22
true_neutral              14             4             33
true_positive             12             0             29


## Augmented Training: Human + LLM Labels

In [14]:
LLM_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)

# Safety: recover holdout IDs from the original ds index and exclude them
holdout_ids = set(ds.loc[ft_holdout_df.index, 'id'])
leaked = llm_labels[llm_labels['id'].isin(holdout_ids)]
if len(leaked):
    print(f"WARNING: {len(leaked)} LLM labels overlap with holdout — dropping them.")
    llm_labels = llm_labels[~llm_labels['id'].isin(holdout_ids)]

# Combine: human train split + all LLM labels (keep company_name for text formatting)
ft_train_augmented_df = pd.concat(
    [ft_train_df, llm_labels[["message", "sentiment", "company_name"]]],
    ignore_index=True,
)

print(f"Train  : {len(ft_train_df)} human  →  {len(ft_train_augmented_df)} total (+{len(llm_labels)} LLM)")
print(f"Holdout: {len(ft_holdout_df)} (unchanged)")
print("\nAugmented train sentiment distribution:")
print(ft_train_augmented_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

Train  : 486 human  →  10486 total (+10000 LLM)
Holdout: 122 (unchanged)

Augmented train sentiment distribution:
sentiment
negative    3711
neutral     2984
positive    3791
Name: count, dtype: int64


In [15]:
# Fine tuned weights as new base — should converge faster and handle more data better than starting from the original base model
model_aug = AutoModelForSequenceClassification.from_pretrained(
    FINETUNED_MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

ft_train_augmented_ds = make_hf_dataset(ft_train_augmented_df)
# ft_holdout_ds is unchanged

AUGMENTED_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/{BASE_MODEL.split("/")[-1]}_augmented_{timestamp}/'

aug_training_args = TrainingArguments(
    output_dir='/tmp/aug_checkpoints/',
    num_train_epochs=15,         # more data needs more epochs to converge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,          # slightly higher — larger dataset can support it
    weight_decay=0.01,
    warmup_steps=250,
    logging_steps=50,            # log training loss every 50 steps so we can see it move
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

aug_trainer = Trainer(
    model=model_aug,
    args=aug_training_args,
    train_dataset=ft_train_augmented_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

aug_trainer.train()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/10486 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.823055,1.051316,0.557377,0.540857,0.650450,0.557377
2,0.733507,0.866872,0.647541,0.637813,0.689318,0.647541
3,0.651674,1.018689,0.631148,0.604942,0.699667,0.631148
4,0.588381,0.975624,0.647541,0.634543,0.722969,0.647541
5,0.495596,1.071462,0.614754,0.608116,0.690910,0.614754
6,0.440888,1.050571,0.688525,0.687269,0.714770,0.688525
7,0.414535,1.064085,0.680328,0.678974,0.720205,0.680328
8,0.405411,1.125378,0.655738,0.655089,0.691290,0.655738
9,0.372461,1.152096,0.655738,0.656258,0.698850,0.655738
10,0.402800,1.180286,0.647541,0.646185,0.685787,0.647541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9840, training_loss=0.4657245195978056, metrics={'train_runtime': 987.9779, 'train_samples_per_second': 159.204, 'train_steps_per_second': 9.96, 'total_flos': 1.0814022828002709e+17, 'train_loss': 0.4657245195978056, 'epoch': 15.0})

In [16]:
aug_trainer.save_model(AUGMENTED_MODEL_PATH)
tokenizer.save_pretrained(AUGMENTED_MODEL_PATH)
print(f"Augmented model saved to: {AUGMENTED_MODEL_PATH}")

aug_results = aug_trainer.predict(ft_holdout_ds)
aug_preds = np.argmax(aug_results.predictions, axis=1)
ft_true   = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — AUGMENTED (human + LLM)")
print("=" * 50)
print(classification_report(ft_true, aug_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, aug_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

# Side-by-side summary vs the human-only run
print("\n── Accuracy comparison ──")
print(f"  Human only  (~486 train) : {accuracy_score(ft_true, ft_preds):.3f}")
print(f"  Human + LLM (~{len(ft_train_augmented_df)} train) : {accuracy_score(ft_true, aug_preds):.3f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Augmented model saved to: /content/drive/MyDrive/ColabThesisData/xlm-roberta-large_augmented_2026-03-19_11-41-38/



HOLDOUT RESULTS — AUGMENTED (human + LLM)
              precision    recall  f1-score   support

    negative       0.54      0.73      0.62        30
     neutral       0.80      0.55      0.65        51
    positive       0.74      0.83      0.78        41

    accuracy                           0.69       122
   macro avg       0.69      0.70      0.68       122
weighted avg       0.71      0.69      0.69       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative             22             5              3
true_neutral              14            28              9
true_positive              5             2             34

── Accuracy comparison ──
  Human only  (~486 train) : 0.328
  Human + LLM (~10486 train) : 0.689
